# Rectified Flow 动漫头像 V2.0：Google Colab T4 训练

本 Notebook 用于从 Google Drive 中的项目 ZIP 启动 V2.0 teacher 训练。V2.0 包含约 62M 参数的扩容 U-Net、16×16/8×8 多头自注意力、EMA 和梯度累积，**不包含 reflow**；reflow 计划在 V2.1 实现。

项目代码与图片会解压到 Colab 的 `/content` 临时磁盘，以提高大量小文件读取速度；checkpoint、生成样本、loss 图、TensorBoard 日志和 FID/KID 指标持久化到：

```text
MyDrive/KRM_RF_Anime_Colab_Results_V2.0
```

> 请先选择“运行时 → 更改运行时类型 → T4 GPU”。V2.0 架构与 V1.x checkpoint 不兼容，不要将 V1.x 权重复制进 V2.0 结果目录。打包 ZIP 时建议排除本地 `outputs/`、`runs/`、`.git/` 和缓存目录，它们不是 Colab 训练必需内容；保留代码、配置、测试、`Data/` 与 `datasets/splits_v1_2/`。

## 1. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 设置 ZIP、结果目录和训练参数

通常只需确认 `DRIVE_ZIP_PATH` 与上传到 Drive 的文件名一致。`TARGET_EPOCHS` 是最终目标 epoch，不是额外训练轮数；断线后保持相同结果目录即可从 V2.0 自己的 `latest.pt` 恢复。

In [ ]:
from pathlib import Path

DRIVE_ZIP_PATH = Path('/content/drive/MyDrive/KRM_RF_Anime Images.zip')
DRIVE_RESULTS_DIR = Path('/content/drive/MyDrive/KRM_RF_Anime_Colab_Results_V2.0')

TARGET_EPOCHS = 120
MICRO_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
EVALUATION_BATCH_SIZE = 16
SAVE_EVERY_EPOCHS = 30
FORCE_REBUILD_SPLITS = False

assert DRIVE_ZIP_PATH.is_file(), f'找不到项目 ZIP：{DRIVE_ZIP_PATH}'
assert 'V2.0' in DRIVE_RESULTS_DIR.name, 'V2.0 必须使用独立结果目录，防止覆盖 V1.x'
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('项目 ZIP：', DRIVE_ZIP_PATH)
print('V2.0 结果目录：', DRIVE_RESULTS_DIR)
print('有效 Batch Size：', MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

## 3. 解压项目到 Colab 临时磁盘

每个新运行时都需要重新解压。这里只清理 `/content/krm_rf_anime_v2_0_workspace`，不会修改 Drive 中的 ZIP、V1.x 结果或 V2.0 持久化结果。

In [ ]:
import shutil
import zipfile

EXTRACT_ROOT = Path('/content/krm_rf_anime_v2_0_workspace')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as archive:
    archive.extractall(EXTRACT_ROOT)

project_candidates = sorted({
    train_file.parent
    for train_file in EXTRACT_ROOT.rglob('train.py')
    if (train_file.parent / 'config' / 'v2_teacher.yaml').is_file()
    and (train_file.parent / 'models' / 'attention.py').is_file()
    and (train_file.parent / 'requirements.txt').is_file()
}, key=lambda path: (len(path.parts), str(path)))
if not project_candidates:
    raise FileNotFoundError('ZIP 中没有找到包含 V2.0 配置和 Attention 实现的项目目录')

PROJECT_DIR = project_candidates[0]
DATA_DIR = PROJECT_DIR / 'Data' / 'train' / 'nolabel'
assert DATA_DIR.is_dir(), f'找不到训练图片目录：{DATA_DIR}'
print('项目目录：', PROJECT_DIR)
print('数据目录：', DATA_DIR)
print('PNG 数量：', len(list(DATA_DIR.glob('*.png'))))

## 4. 检查 T4 GPU 和 Colab 环境

In [ ]:
import platform
import subprocess
import sys
import torch

print('Python：', sys.version.split()[0])
print('系统：', platform.platform())
print('PyTorch：', torch.__version__)
print('CUDA 可用：', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('当前没有 GPU，请切换到 T4 GPU 运行时后重新连接')
gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('GPU：', gpu_name)
print('GPU 显存：', round(gpu_memory, 2), 'GiB')
if 'T4' not in gpu_name:
    print('提示：当前不是 T4，但只要 CUDA 显存充足仍可继续。')
subprocess.run(['nvidia-smi'], check=True)

## 5. 安装 V2.0 依赖

保留 Colab 自带的 CUDA 版 PyTorch/torchvision，只安装项目依赖。Attention 使用 PyTorch 原生 SDPA，不需要额外注意力库。

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')],
    check=True,
)

import imagehash
import torch_fidelity
import torchmetrics
import torchvision
import yaml

print('torch：', torch.__version__)
print('torchvision：', torchvision.__version__)
print('torchmetrics：', torchmetrics.__version__)
print('ImageHash：', imagehash.__version__)

## 6. 生成 Drive 专用 V2.0 配置

读取 `config/v2_teacher.yaml`，只覆盖 Colab 路径和运行参数。原始图片从 `/content` 读取，所有 V2.0 训练产物写入独立 Drive 目录。

In [ ]:
with (PROJECT_DIR / 'config' / 'v2_teacher.yaml').open('r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

config['project']['name'] = 'rectified-flow-anime-v2.0-colab-t4'
config['data']['raw_dir'] = str(DATA_DIR)
config['data']['split_dir'] = str(DRIVE_RESULTS_DIR / 'datasets' / 'splits_v1_2')
config['data']['auto_prepare'] = False
config['data']['num_workers'] = 2
config['training']['device'] = 'cuda'
config['training']['epochs'] = TARGET_EPOCHS
config['training']['batch_size'] = MICRO_BATCH_SIZE
config['training']['gradient_accumulation_steps'] = GRADIENT_ACCUMULATION_STEPS
config['training']['mixed_precision'] = True
config['training']['save_every_epochs'] = SAVE_EVERY_EPOCHS
config['training']['resume'] = True
config['evaluation']['batch_size'] = EVALUATION_BATCH_SIZE

config['paths']['output_dir'] = str(DRIVE_RESULTS_DIR)
config['paths']['checkpoint_dir'] = str(DRIVE_RESULTS_DIR / 'checkpoints')
config['paths']['sample_dir'] = str(DRIVE_RESULTS_DIR / 'samples')
config['paths']['plot_dir'] = str(DRIVE_RESULTS_DIR / 'plots')
config['paths']['evaluation_dir'] = str(DRIVE_RESULTS_DIR / 'evaluation')
config['paths']['log_dir'] = str(DRIVE_RESULTS_DIR / 'runs')

COLAB_CONFIG = DRIVE_RESULTS_DIR / 'config_colab_v2_0.yaml'
with COLAB_CONFIG.open('w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)

for directory in config['paths'].values():
    Path(directory).mkdir(parents=True, exist_ok=True)

print('Colab 配置：', COLAB_CONFIG)
print('目标 epoch：', config['training']['epochs'])
print('Micro Batch：', config['training']['batch_size'])
print('梯度累积：', config['training']['gradient_accumulation_steps'])
print('有效 Batch：', config['training']['batch_size'] * config['training']['gradient_accumulation_steps'])
print('Attention 分辨率：', config['model']['attention_resolutions'])
print('历史快照间隔：', config['training']['save_every_epochs'], 'epochs')
print('采样器：', config['sampling']['solver'], config['sampling']['num_steps'], '步')

## 7. 运行单元测试

检查 Attention、U-Net skip connection、梯度累积、V1/V2 输出隔离和 Rectified Flow。测试失败时不要继续训练。

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=PROJECT_DIR,
    check=True,
)

## 8. 准备固定的数据清单

默认优先复制 ZIP 内的 `datasets/splits_v1_2` 到 V2.0 Drive 目录，确保和 V1.2 使用同一训练/验证/测试划分。只有设置 `FORCE_REBUILD_SPLITS=True` 才重新扫描图片并执行 pHash。该操作不会删除原始图片。

In [ ]:
import json

split_dir = Path(config['data']['split_dir'])
packaged_split_dir = PROJECT_DIR / 'datasets' / 'splits_v1_2'
required_manifests = ('train.txt', 'val.txt', 'test.txt', 'metadata.json')

if FORCE_REBUILD_SPLITS:
    split_command = [
        sys.executable, '-m', 'datasets.split_dataset',
        '--config', str(COLAB_CONFIG), '--force',
    ]
    subprocess.run(split_command, cwd=PROJECT_DIR, check=True)
elif all((split_dir / name).is_file() for name in required_manifests):
    print('复用 Drive 中已有的固定数据清单。')
elif all((packaged_split_dir / name).is_file() for name in required_manifests):
    split_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(packaged_split_dir, split_dir, dirs_exist_ok=True)
    print('已把 ZIP 内的 V1.2 固定清单复制到 V2.0 Drive 目录。')
else:
    split_command = [
        sys.executable, '-m', 'datasets.split_dataset',
        '--config', str(COLAB_CONFIG),
    ]
    subprocess.run(split_command, cwd=PROJECT_DIR, check=True)

metadata = json.loads((split_dir / 'metadata.json').read_text(encoding='utf-8'))
print('原始图片：', metadata['total_files'])
print('完全重复：', metadata['exact_duplicate_files'])
print('近似重复：', metadata['near_duplicate_files'])
print('最终保留：', metadata['unique_images'])
print('数据划分：', metadata['splits'])
print('清单目录：', split_dir)

## 9. 启动 TensorBoard（可选）

可以在训练前执行。日志直接写入并读取 V2.0 的 Drive 持久化目录。

In [ ]:
TENSORBOARD_LOG_DIR = str(DRIVE_RESULTS_DIR / 'runs')
%load_ext tensorboard
%tensorboard --logdir $TENSORBOARD_LOG_DIR --port 6006

## 10. V2.0 模型与 T4 显存 smoke test

构建完整模型、EMA 和 Adam，执行一次 micro-batch 前向/反向/更新并报告峰值显存。测试对象随后删除，不会保存 checkpoint，也不会影响正式训练。

In [ ]:
import gc
import os

project_path = str(PROJECT_DIR)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
os.chdir(PROJECT_DIR)

from flow import RectifiedFlow
from models import AttentionBlock
from utils.config import load_config
from utils.ema import ExponentialMovingAverage
from utils.model import build_model

smoke_config = load_config(COLAB_CONFIG)
device = torch.device('cuda')
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
smoke_model = build_model(smoke_config, device)
smoke_ema = ExponentialMovingAverage(smoke_model, decay=0.9999)
smoke_flow = RectifiedFlow(smoke_model)
smoke_optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=2e-4)
smoke_scaler = torch.amp.GradScaler('cuda', enabled=True)
smoke_images = torch.randn(MICRO_BATCH_SIZE, 3, 64, 64, device=device)
smoke_optimizer.zero_grad(set_to_none=True)
with torch.autocast(device_type='cuda', dtype=torch.float16):
    smoke_loss = smoke_flow.training_loss(smoke_images) / GRADIENT_ACCUMULATION_STEPS
smoke_scaler.scale(smoke_loss).backward()
smoke_scaler.unscale_(smoke_optimizer)
smoke_scaler.step(smoke_optimizer)
smoke_scaler.update()
smoke_ema.update(smoke_model)

parameter_count = sum(parameter.numel() for parameter in smoke_model.parameters())
attention_count = sum(isinstance(module, AttentionBlock) for module in smoke_model.modules())
print('参数量：', f'{parameter_count / 1_000_000:.2f}M')
print('AttentionBlock：', attention_count)
print('Smoke loss：', float(smoke_loss.detach()) * GRADIENT_ACCUMULATION_STEPS)
print('峰值已分配显存：', round(torch.cuda.max_memory_allocated() / 1024**3, 2), 'GiB')
print('峰值保留显存：', round(torch.cuda.max_memory_reserved() / 1024**3, 2), 'GiB')

del smoke_images, smoke_loss, smoke_flow, smoke_optimizer, smoke_scaler, smoke_ema, smoke_model
gc.collect()
torch.cuda.empty_cache()

## 11. 开始或继续训练

该单元格借鉴 V1.2，在当前 Colab 会话中临时把训练脚本的 tqdm 替换为 Notebook 原生进度条，不修改 `train.py`。外层显示总 Epoch 与恢复位置，内层显示训练/验证 micro-batch、实时 loss 和预计剩余时间；外层 postfix 显示 epoch train loss、validation loss 和学习率。

每个 epoch 会原子更新 Drive 中 V2.0 的 `latest.pt`，最佳验证结果更新 `best.pt`；断线后重新执行第 1～8、10、11 节即可恢复。V2.0 checkpoint 接近 1 GiB，写入 Drive 时进度条短暂停顿属于正常现象。

In [ ]:
import builtins
import os
import sys
import time
from IPython.display import Markdown, display
from tqdm.notebook import tqdm as notebook_tqdm

project_path = str(PROJECT_DIR)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
os.chdir(PROJECT_DIR)

import train as train_module
from utils.config import load_config

original_tqdm = train_module.tqdm
original_train_one_epoch = train_module.train_one_epoch
original_evaluate_loss = train_module.evaluate_loss
progress_state = {'epoch_bar': None, 'train': None, 'val': None, 'lr': None}

def colab_tqdm(iterable=None, *args, **kwargs):
    kwargs.setdefault('mininterval', 0.5)
    return notebook_tqdm(iterable, *args, **kwargs)

def colab_epoch_range(*args):
    values = builtins.range(*args)
    start_epoch = values.start
    target_epoch = max(values.stop - 1, 0)
    epoch_bar = notebook_tqdm(
        values,
        desc='V2.0 Epoch',
        unit='epoch',
        initial=max(start_epoch - 1, 0),
        total=target_epoch,
    )
    progress_state['epoch_bar'] = epoch_bar
    return epoch_bar

def colab_train_one_epoch(*args, **kwargs):
    train_loss, global_step = original_train_one_epoch(*args, **kwargs)
    progress_state['train'] = train_loss
    epoch_bar = progress_state['epoch_bar']
    optimizer = kwargs.get('optimizer')
    if epoch_bar is not None and optimizer is not None:
        progress_state['lr'] = optimizer.param_groups[0]['lr']
        epoch_bar.set_postfix(
            train=f'{train_loss:.4f}',
            lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            refresh=True,
        )
    return train_loss, global_step

def colab_evaluate_loss(*args, **kwargs):
    val_loss = original_evaluate_loss(*args, **kwargs)
    progress_state['val'] = val_loss
    epoch_bar = progress_state['epoch_bar']
    if epoch_bar is not None:
        postfix = {'val': f'{val_loss:.4f}'}
        if progress_state['train'] is not None:
            postfix['train'] = f"{progress_state['train']:.4f}"
        if progress_state['lr'] is not None:
            postfix['lr'] = f"{progress_state['lr']:.2e}"
        epoch_bar.set_postfix(postfix, refresh=True)
    return val_loss

latest_checkpoint = DRIVE_RESULTS_DIR / 'checkpoints' / 'latest.pt'
resume_message = '从头训练'
if latest_checkpoint.is_file():
    resume_message = f'将从 {latest_checkpoint.name} 恢复'
display(Markdown(
    f'**V2.0：{TARGET_EPOCHS} epochs · micro-batch {MICRO_BATCH_SIZE} · '
    f'累积 {GRADIENT_ACCUMULATION_STEPS} · 有效 batch '
    f'{MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS} · {resume_message}**'
))
training_started = time.time()
try:
    train_module.range = colab_epoch_range
    train_module.tqdm = colab_tqdm
    train_module.train_one_epoch = colab_train_one_epoch
    train_module.evaluate_loss = colab_evaluate_loss
    train_module.run_training(load_config(COLAB_CONFIG))
finally:
    if hasattr(train_module, 'range'):
        delattr(train_module, 'range')
    train_module.tqdm = original_tqdm
    train_module.train_one_epoch = original_train_one_epoch
    train_module.evaluate_loss = original_evaluate_loss
elapsed_minutes = (time.time() - training_started) / 60
display(Markdown(f'✅ **训练单元格执行完成，用时 {elapsed_minutes:.2f} 分钟。**'))

## 12. 查看 checkpoint、loss 图和最新样本

In [ ]:
from IPython.display import display
from PIL import Image

checkpoint_dir = DRIVE_RESULTS_DIR / 'checkpoints'
for checkpoint_path in sorted(checkpoint_dir.glob('*.pt')):
    print(checkpoint_path.name, round(checkpoint_path.stat().st_size / 1024**2, 1), 'MiB')

loss_curve = DRIVE_RESULTS_DIR / 'plots' / 'loss_curve.png'
if loss_curve.is_file():
    display(Image.open(loss_curve))

sample_paths = sorted((DRIVE_RESULTS_DIR / 'samples').glob('epoch_*.png'))
if sample_paths:
    print('最新训练样本：', sample_paths[-1])
    display(Image.open(sample_paths[-1]))

## 13. 使用最佳 EMA 权重独立生成图片

`sample.py` 默认优先加载 EMA 权重。修改 `SAMPLE_SEED` 可以生成不同图片。

In [ ]:
from IPython.display import display
from PIL import Image

SAMPLE_SEED = 123
BEST_CHECKPOINT = DRIVE_RESULTS_DIR / 'checkpoints' / 'best.pt'
assert BEST_CHECKPOINT.is_file(), f'找不到 V2.0 最佳模型：{BEST_CHECKPOINT}'
INDEPENDENT_SAMPLE = DRIVE_RESULTS_DIR / 'samples' / f'ema_seed_{SAMPLE_SEED}.png'

subprocess.run(
    [
        sys.executable, 'sample.py',
        '--config', str(COLAB_CONFIG),
        '--checkpoint', str(BEST_CHECKPOINT),
        '--seed', str(SAMPLE_SEED),
        '--output', str(INDEPENDENT_SAMPLE),
    ],
    cwd=PROJECT_DIR,
    check=True,
)
display(Image.open(INDEPENDENT_SAMPLE))

## 14. 计算固定 flow MSE、FID 和 KID

默认使用 V2.0 最佳 EMA 权重、固定验证随机数、固定生成噪声和 Heun 50 步。首次运行 torch-fidelity 时会下载 Inception 权重。结果写入 V2.0 Drive 目录的 `evaluation/metrics.json`，不会覆盖 V1.2 指标。

In [ ]:
import json

subprocess.run(
    [
        sys.executable, 'evaluate.py',
        '--config', str(COLAB_CONFIG),
        '--checkpoint', str(BEST_CHECKPOINT),
    ],
    cwd=PROJECT_DIR,
    check=True,
)

metrics_path = DRIVE_RESULTS_DIR / 'evaluation' / 'metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## Colab 断线恢复与空间建议

- 新运行时重新执行第 1～8 节，将 ZIP 解压到新的 `/content`；第 10 节 smoke test 可再次执行，也可在已验证后跳过。
- 结果目录不变时，第 11 节自动读取 V2.0 `checkpoints/latest.pt` 恢复。
- `TARGET_EPOCHS` 是最终目标。例如 checkpoint 已到 epoch 40，设置 120 会继续到 120。
- 默认 micro-batch 8、累积 4 次。若意外 OOM，先改成 micro-batch 4、累积 8 次，有效 batch 仍为 32，模型结构和 checkpoint 兼容。
- V2.0 checkpoint 接近 1 GiB；`best.pt`、`latest.pt` 加每 30 epoch 的历史快照预计占用约 6 GiB。Drive 空间不足时可以增大 `SAVE_EVERY_EPOCHS`，不要删除正在写入的 `.tmp` 文件。
- 不要把 V1.x checkpoint 放进 `KRM_RF_Anime_Colab_Results_V2.0/checkpoints`。
- 只有更改数据或 pHash 参数时才设置 `FORCE_REBUILD_SPLITS=True`；完成后改回 `False`。
- V1.2 基线为 FID 36.4685、KID 0.015611 ± 0.000764、flow MSE 0.191947；V2.0 应保持相同清单和评估 seed 对比。